# Run a planner design on qBraid

*Companion to the QKD Grid Optimizer page (`qkd_grid_optimizer.html`).* The planner chooses which links get QKD. This notebook takes that design and runs **BB84 on every QKD link at once, in one circuit, on qBraid's GPU simulator**, with an eavesdropper on the busiest QKD link. It's the check that each chosen link produces a matching key and that the alarm fires where the attacker is.

**Steps:** in the planner, press **Download design JSON**, upload `grid_design.json` to this folder (a default Chattanooga design is used if none is found), then run all cells.

> **Kernel:** qBraid-SDK (v0.12.2). The GPU simulator bills about 5 credits per minute; this job takes seconds.

In [ ]:
import importlib.util, json, os, sys, numpy as np
if importlib.util.find_spec("qbraid") is None:
    raise ImportError("Wrong kernel: pick qBraid-SDK (v0.12.2) at the top right, then run this cell again.")
from qiskit import QuantumCircuit
sys.path.insert(0, "network_design")
path = "grid_design.json" if os.path.exists("grid_design.json") else "network_design/grid_design_default.json"
design = json.load(open(path))
links = design["qkd_links"]
print(f"Design from {path}: {len(links)} QKD links, trusted nodes {design['trusted_nodes']}")
for L in links: print(f"   {L['a']} - {L['b']}: {L['km']} km, {L['loss_db']} dB, {L['traffic_mw']} MW of control traffic")

## 1. One circuit for the whole design

Each QKD link gets `ROUNDS` BB84 rounds on its own qubits. Only sifted rounds are simulated (Alice's and Bob's bases match), since those are the ones that become key. Eve guesses her basis wrong on 2 of the 3 tapped rounds, close to the 50% she would get by chance. Eve taps every round of the **busiest** QKD link (the one carrying the most control traffic), with one extra qubit per tapped round, as in `3a` and `3f`. Qubit count = links × rounds + rounds, kept within the GPU simulator's 35 qubits.

In [ ]:
ROUNDS = max(1, min(3, (35 - 3) // max(1, len(links))))
rng = np.random.default_rng(11)
target = max(range(len(links)), key=lambda i: links[i]["traffic_mw"])
nq = len(links) * ROUNDS + ROUNDS
qc = QuantumCircuit(nq, len(links) * ROUNDS)
plan = []                      # (link index, qubit, alice bit, alice basis, bob basis, eve basis or None)
eve_q = len(links) * ROUNDS
for i in range(len(links)):
    for r in range(ROUNDS):
        q = i * ROUNDS + r; a, ab = (int(x) for x in rng.integers(0, 2, 2)); bb = ab; eb = None   # only sifted rounds are simulated (bases match)
        used = False
        if a: qc.x(q); used = True
        if ab: qc.h(q); used = True
        if i == target:
            eb = ab ^ (1 if r % 2 == 0 else 0)     # Eve guesses wrong on rounds 0 and 2, right on round 1
            if eb: qc.h(q)
            qc.cx(q, eve_q); eve_q += 1; used = True
            if eb: qc.h(q)
        if bb: qc.h(q); used = True
        if not used: qc.id(q)
        plan.append((i, q, int(a), int(ab), int(bb), eb))
qc.measure(range(len(links) * ROUNDS), range(len(links) * ROUNDS))
print(f"{nq} qubits, {ROUNDS} rounds per link, eavesdropper on {links[target]['a']} - {links[target]['b']}")

## 2. Run on qBraid's GPU simulator

In [ ]:
from qbraid.runtime import QbraidProvider
import warnings; warnings.filterwarnings("ignore", message="Circuit contains measurement gates")
DEVICE = "qbraid:qbraid:sim:gpu-sv"          # fallback if the GPU queue is busy: "qbraid:qbraid:sim:qir-sv" (30 qubits)
SHOTS = 500
job = QbraidProvider().get_device(DEVICE).run(qc, shots=SHOTS)
counts = job.result().data.get_counts()
print("Job ID:", job.id, "| status:", job.status())

## 3. Per-link result

In [ ]:
R0 = 3200     # bps at 0 dB, fitted to EPB's measured links (Evans et al., 2021)
rows = []
for i, L in enumerate(links):
    err = tot = 0
    for bs, c in counts.items():
        bits = bs.replace(" ", "")[::-1]
        for (li, q, a, ab, bb, eb) in plan:
            if li != i or ab != bb: continue
            err += c * (int(bits[q]) != a); tot += c
    qber = err / tot if tot else float("nan")
    rows.append((L, qber, tot // SHOTS))
    flag = "ALARM (tap)" if qber > 0.11 else ("ok" if tot else "no sifted rounds")
    print(f"{L['a']:>16} - {L['b']:<16} QBER {qber:6.3f}  sifted rounds {tot // SHOTS}  expected key rate ~{R0 * 10 ** (-L['loss_db'] / 10):,.0f} bps  {flag}")
print("\nRun record:", f"3h design check | {DEVICE} | {job.id} | {SHOTS} | {job.status()}")

**Reading it:** every link without an eavesdropper should show QBER near 0, so its key matches. The tapped link should show a high QBER and trip the alarm. With only a few rounds per link, a link whose rounds all landed in mismatched bases has no sifted rounds; rerun with a different seed for more. The key-rate column comes from the loss model, not the circuit. It shows each chosen link stays well above the 4.3 bps needed to refresh an AES-256 key every minute.